In [1]:
import os
import json
import pandas as pd

from pathlib import Path


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 53 samples and 11449 columns.
First few rows of the data:


,OS (month),Vital status,DFS (month),Disease-free,Sample,Op,Pathology,Risk factor,Progression,Tumor stage,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,46,0,46,0,DH013,TH,DN high,HBV,NaN,Stage II,...,7.452408,3.620973,6.167827,8.543239,5.840271,5.237661,6.757061,6.641830,4.346114,6.949632
1,102,0,102,0,TG1-003,TH,ES1,HBV,NaN,Stage II,...,7.336706,3.639574,6.317031,7.831987,5.338136,5.270831,6.924206,6.534365,3.967166,6.925958
2,119,0,119,0,TG1-005,TH,ES1,HBV,NaN,Stage II,...,6.934483,3.769259,6.098319,7.832153,5.097037,5.028956,6.750284,6.649036,4.280020,6.924831
3,91,0,91,0,TG1-007,TH,ES1,HCV,NaN,Stage II,...,7.091694,3.916144,6.154706,7.928919,5.603064,5.310771,6.782520,6.695312,4.229381,7.131261
4,84,0,84,0,TG1-009,TH,ES1,HBV,NaN,Stage I,...,7.750944,4.205411,6.733334,8.155534,6.379199,5.685522,7.172040,6.495033,4.414602,6.813365


In [2]:
n_stats_written = 0

for idx, endpoint in enumerate(summary_data['survival-endpoints']):
    time_var = endpoint['time_var']['var_name']
    event_var = endpoint['event_var']['var_name']

    if time_var not in df.columns or event_var not in df.columns:
        abbrv = endpoint.get('abbrv', f'endpoint[{idx}]')
        print(
            f"Skipping stats for {abbrv}: columns not in preprocessed table "
            f"({time_var!r}, {event_var!r})."
        )
        endpoint.pop('stats', None)
        continue

    complete_mask = df[[time_var, event_var]].notnull().all(axis=1)
    n_complete = int(complete_mask.sum())
    n_incomplete = int(len(df) - n_complete)

    # Among samples with complete time + event data
    censored_count = int((complete_mask & (df[event_var] == 0.0)).sum())
    event_count = int((complete_mask & (df[event_var] == 1.0)).sum())

    censored_ratio = censored_count / n_complete

    print(f"Number of samples with incomplete data for {time_var} or {event_var}: {n_incomplete}")
    print(f"Number of samples with complete data for both variables: {n_complete}")
    print(f"Number of censored samples (event_var == 0.0): {censored_count}")
    print(f"Number of events (event_var == 1.0): {event_count}")
    pct = 100.0 * censored_ratio
    print(
        f"Censored ratio (among complete): {censored_count} / {n_complete} = "
        f"{censored_ratio:.3f} ({pct:.1f}%)"
    )

    # Refresh only this block on each run; leave all other endpoint fields unchanged
    endpoint['stats'] = {
        'n_incomplete': n_incomplete,
        'n_complete': n_complete,
        'n_censored': censored_count,
        'n_events': event_count,
        'censored_ratio': round(censored_ratio, 2),
    }
    n_stats_written += 1

with open(description_path, 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, indent=4, ensure_ascii=False)
    f.write('\n')

print(
    f"Updated stats for {n_stats_written} of {len(summary_data['survival-endpoints'])} endpoint(s); "
    f"wrote {description_path.resolve()}"
)


Number of samples with incomplete data for OS (month) or Vital status: 0
Number of samples with complete data for both variables: 53
Number of censored samples (event_var == 0.0): 47
Number of events (event_var == 1.0): 6
Censored ratio (among complete): 47 / 53 = 0.887 (88.7%)
Number of samples with incomplete data for DFS (month) or Disease-free: 0
Number of samples with complete data for both variables: 53
Number of censored samples (event_var == 0.0): 32
Number of events (event_var == 1.0): 21
Censored ratio (among complete): 32 / 53 = 0.604 (60.4%)
Updated stats for 2 of 2 endpoint(s); wrote /Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/geo/GSE148355/description.json
